In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from openai import OpenAI
from autoddg import AutoDDG
from autoddg.utils import get_sample
from autoddg.evaluation import BaseEvaluator
from typing import Optional
# --- Import custom files ---
from prompts import ALL_RELATED_WORK_PROMPTS
from utils import log_result, run_description_experiment
import os 
import json
from cache_utils import run_with_caching, load_profile_from_cache#, MockAutoDDG           #Mock is for testing

[SETUP] Project Root: /home/bia/Documents/AutoDDG-Enhanced
[SETUP] Cache Directory: /home/bia/Documents/AutoDDG-Enhanced/prompt-experiments/profile_cache


In [2]:
# --- LLM Config ---
MODEL_CONFIG = {
    "base_url": "http://localhost:11434/v1",
    "api_key": "ollama",
    "model_name": "llama3.1:8b",
}

# --- Experiment Config ---
DATABASE_PATH_ = '../src/autoddg/database.json'
RESULTS_FILE_RELATIVE = 'results.csv' # Keep the relative part
PROFILE_CACHE_DIR = 'profile_cache'

# Define necessary base directories
script_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
DATABASE_PATH = os.path.join(script_dir, DATABASE_PATH_)
PROJECT_ROOT = os.path.abspath(os.path.join(script_dir, os.pardir))

# --- NEW: Define the ABSOLUTE path for the results file ---
# We assume the results file path is relative to the directory where the current script is.
RESULTS_FILE_ABSOLUTE = os.path.join(script_dir, RESULTS_FILE_RELATIVE)

# Rename the variable used in your logging
RESULTS_FILE = RESULTS_FILE_ABSOLUTE

# --- Define Evaluation Class ---
class Eval(BaseEvaluator):
    def __init__(self, model_name: str = MODEL_CONFIG["model_name"]):
        client = OpenAI(
            api_key=MODEL_CONFIG["api_key"], 
            base_url=MODEL_CONFIG["base_url"]
        )
        super().__init__(client=client, model_name=model_name)

# Initialize Core Tools
client = OpenAI(api_key=MODEL_CONFIG["api_key"], base_url=MODEL_CONFIG["base_url"])
auto_ddg = AutoDDG(client=client, model_name=MODEL_CONFIG["model_name"])
auto_ddg.set_evaluator(Eval())

In [3]:
with open(DATABASE_PATH, 'r') as f:
    raw_database = json.load(f)
    # Convert keys to integers if they are dataset IDs
    database = {int(k): v for k, v in raw_database.items()}

# 2. Get the first dataset entry
try:
    # Use next(iter()) to reliably get the first key/value pair.
    # The key is the dataset_id, and the value (the info dictionary) is dataset_info.
    dataset_id, dataset_info = next(iter(database.items()))
except StopIteration:
    print("Error: The database file is empty.")
    exit()

# dataset_info is now the full metadata dictionary (e.g., {'dataset_name': '...', 'description': '...'}).

print(f"Running experiment on first dataset: ID='{dataset_id}'")

# 3. Call the run_with_caching function
run_with_caching(dataset_id, dataset_info, auto_ddg)

Running experiment on first dataset: ID='3222451'

[RUNNER] Attempting to process The FluPRINT database...
[CACHE] Loaded profiles for 3222451 from cache.
[RUNNER] Successfully retrieved profiles for The FluPRINT database.
  Topic: It appears to be a dataset related to "Immune Cell Analysis" or "Cytokine Profiling".
--------------------------------------------------


In [4]:
# Define which prompts you want to run for this experiment
PROMPTS_TO_TEST = {
    "Structured_v1-llm-relevance-chunks-new-integration": ALL_RELATED_WORK_PROMPTS["Structured_v1"],
    # "V2_Aggressive": ALL_RELATED_WORK_PROMPTS["V2_Aggressive"],
    # "V2_Hybrid": ALL_RELATED_WORK_PROMPTS["V2_Hybrid"]
}
print(f"Testing {len(PROMPTS_TO_TEST)} related work prompts.")



Testing 1 related work prompts.


In [5]:
print("--- Starting Full Experiment Run ---")

# Iterate through every entry in your database
for dataset_id, dataset_info in database.items():
    # Attempt to load the profile from cache using the ID
    cached_profiles = load_profile_from_cache(dataset_id)
    
    if cached_profiles is not None:
        # If the profile exists in the cache, run the experiment
        run_description_experiment(
            dataset_id=dataset_id,
            dataset_info=dataset_info,
            auto_ddg=auto_ddg,
            PROJECT_ROOT=PROJECT_ROOT,
            RESULTS_FILE=RESULTS_FILE,
            PROMPTS_TO_TEST=PROMPTS_TO_TEST,
            load_profile_from_cache=load_profile_from_cache,
            log_result=log_result
        )
    else:
        # Skip the dataset if the cache is missing
        print(f"SKIP: Dataset ID {dataset_id} ('{dataset_info.get('dataset_name', 'Unknown')}') is missing cache profile and will be skipped.")

print("\nAll experiments complete. Check results in:", RESULTS_FILE)

--- Starting Full Experiment Run ---
[CACHE] Loaded profiles for 3222451 from cache.

✨ Starting Experiment for Dataset: ID='3222451', Name='The FluPRINT database'
[CACHE] Loaded profiles for 3222451 from cache.
Setup complete. PDF: /home/bia/Documents/AutoDDG-Enhanced/src/autoddg/related/papers/3222451.pdf

--- Running Augmented (AutoDDG + Related Work) Descriptions ---

-> Augmented Test with Prompt: Structured_v1-llm-relevance-chunks-new-integration
Reading PDF from: /home/bia/Documents/AutoDDG-Enhanced/src/autoddg/related/papers/3222451.pdf
Successfully extracted text from 20 pages (total: 20 pages)
Total characters extracted: 91565
Original text split into 60 chunks.
Extracting profile for dataset: The FluPRINT database
Sending 65807 characters of CONTEXT to LLM...
Successfully extracted profile (2390 characters)
Related Work Summary: This is a research article published in the journal Circulation, titled "Pathogenic Immune Cell Subsets in ICI Myocarditis". The article discusses t

Ignoring wrong pointing object 43 0 (offset 0)


Augmented Scores (Structured_v1-llm-relevance-chunks-new-integration): Completeness: 9
Conciseness: 8
Readability: 9
Logged Augmented_AutoDDG with Prompt Structured_v1-llm-relevance-chunks-new-integration to /home/bia/Documents/AutoDDG-Enhanced/prompt-experiments/results.csv

✅ Experiment complete for ID 3222451.

[CACHE] Loaded profiles for 4916206 from cache.

✨ Starting Experiment for Dataset: ID='4916206', Name='CODE-15%: a large scale annotated dataset of 12-lead ECGs'
[CACHE] Loaded profiles for 4916206 from cache.
Setup complete. PDF: /home/bia/Documents/AutoDDG-Enhanced/src/autoddg/related/papers/code15.pdf

--- Running Augmented (AutoDDG + Related Work) Descriptions ---

-> Augmented Test with Prompt: Structured_v1-llm-relevance-chunks-new-integration
Reading PDF from: /home/bia/Documents/AutoDDG-Enhanced/src/autoddg/related/papers/code15.pdf
Successfully extracted text from 10 pages (total: 10 pages)
Total characters extracted: 55673
Original text split into 34 chunks.
Extrac

In [14]:
dataset_id = 7651129
# Use .get() for safe lookup, which returns None if the key is not found
dataset_info = database.get(dataset_id) 

if dataset_info is None:
    print(f"Error: Dataset ID '{dataset_id}' not found in the database.")
    exit()

# If the ID is found, you can now proceed to call your runner function
print(f"Running experiment on selected dataset: ID='{dataset_id}', Name='{dataset_info['dataset_name']}'")

DATASET_NAME = dataset_info['dataset_name'] 

# 2. Define the PAPER_FILE path
# Use the correct key: 'related_paper_path'
PAPER_FILE_RELATIVE = dataset_info['related_paper_path'] 

# Resolve the absolute path needed by auto_ddg.analyze_related()
# Note: PROJECT_ROOT must be defined earlier in your script.
PAPER_FILE = os.path.join(PROJECT_ROOT, PAPER_FILE_RELATIVE) 

profiles = load_profile_from_cache(dataset_id=dataset_id)
# --- Unpack the Profiles (Assuming 'profiles' dict is loaded from cache) ---
basic_profile = profiles["basic_profile"]
semantic_profile = profiles["semantic_profile"]
data_topic = profiles["data_topic"]
dataset_sample = profiles["dataset_sample"] 

print(f"Experiment setup complete for: {DATASET_NAME}")
print(f"PDF location set to: {PAPER_FILE}")

Running experiment on selected dataset: ID='7651129', Name='Coronavirus disease (COVID-19) case data - South Africa'
[CACHE] Loaded profiles for 7651129 from cache.
Experiment setup complete for: Coronavirus disease (COVID-19) case data - South Africa
PDF location set to: /home/bia/Documents/AutoDDG-Enhanced/src/autoddg/related/papers/elife-78933-v1.pdf
